In [1]:
"""
Step 1: Get the topological order of DAGs.
Step 2: Generate the scenario.
Step 3: Run the three algorithms and compare the results.
"""
from models.utils.dataset import *
from models.utils.scenario import *
from models.scheduler.dpe import DPE
from models.scheduler.fixdoc import FixDoc
from models.scheduler.heft import HEFT
from models.utils.interpretate_result import *
import datetime

E:\Workshop\real-time-faas\models\scheduler\heft.py:229: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if funcs_num is ():


In [2]:
print('------------------------ Step 1 ------------------------')
sample_jobs()
get_topological_order()

------------------------ Step 1 ------------------------
Dataset (batch_task.csv & batch_instance.csv) is already selected.
Jobs' topological order has been obtained.


In [3]:
print('------------------------ Step 2 ------------------------')
G, bw, pp = generate_scenario()
print_scenario(G, bw, pp)
simple_paths = get_simple_paths(G)
print_simple_paths(simple_paths)
reciprocals_list, proportions_list = get_ratio(simple_paths, bw)
pp_required, data_stream = set_funcs()

------------------------ Step 2 ------------------------

The connected graph of edge servers (represented by adjacent matrix):
array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

====> throughput of each link <====
array([[-1., 49., 44., 49.],
       [49., -1., 35., 66.],
       [44., 35., -1., 54.],
       [49., 66., 54., -1.]])

====> processing power of edge server <====
array([ 8,  8, 10, 10])

====> All simple paths between any two server <====
[[[[0]],
  [[0, 1], [0, 2, 1], [0, 2, 3, 1], [0, 3, 1], [0, 3, 2, 1]],
  [[0, 1, 2], [0, 1, 3, 2], [0, 2], [0, 3, 1, 2], [0, 3, 2]],
  [[0, 1, 2, 3], [0, 1, 3], [0, 2, 1, 3], [0, 2, 3], [0, 3]]],
 [[[1, 0], [1, 2, 0], [1, 2, 3, 0], [1, 3, 0], [1, 3, 2, 0]],
  [[1]],
  [[1, 0, 2], [1, 0, 3, 2], [1, 2], [1, 3, 0, 2], [1, 3, 2]],
  [[1, 0, 2, 3], [1, 0, 3], [1, 2, 0, 3], [1, 2, 3], [1, 3]]],
 [[[2, 0], [2, 1, 0], [2, 1, 3, 0], [2, 3, 0], [2, 3, 1, 0]],
  [[2, 0, 1], [2, 0, 3, 1], [2, 1], [2, 3

In [4]:
print('------------------------ Step 3 ------------------------')
dpe = DPE(G, bw, pp, simple_paths, reciprocals_list, proportions_list, pp_required, data_stream)
start = datetime.datetime.now()
T_optimal_all_dpe, DAGs_deploy_dpe, process_sequence_all_dpe, start_time_all_dpe = dpe.get_response_time(sorted_DAG_path=BATCH_TASK_TOPOLOGICAL_ORDER_PATH)
end = datetime.datetime.now()
print('Computer\'s running time:', (end - start).seconds, 'seconds')
DAG_num_chosen = 2010    # 随机选择采样出来的 DAG 中的某一个
# show_DAG()  # TODO
print_scheduling_results(T_optimal_all_dpe, DAGs_deploy_dpe, process_sequence_all_dpe, start_time_all_dpe, DAG_num_chosen)

------------------------ Step 3 ------------------------

Getting makespan for 219 DAGs by DPE algorithm ...
2% [#.................................................]

ValueError: invalid literal for int() with base 10: 'ask'

In [ ]:
fixdoc = FixDoc(G, bw, pp, simple_paths, reciprocals_list, proportions_list, pp_required, data_stream)
start = datetime.datetime.now()
T_optimal_all_fixdoc, DAGs_deploy_fixdoc, process_sequence_all_fixdoc, start_time_all_fixdoc = fixdoc.get_response_time(sorted_DAG_path=BATCH_TASK_TOPOLOGICAL_ORDER_PATH)
end = datetime.datetime.now()
print('Computer\'s running time:', (end - start).seconds, 'seconds')
print_scheduling_results(T_optimal_all_fixdoc, DAGs_deploy_fixdoc, process_sequence_all_fixdoc, start_time_all_fixdoc, DAG_num_chosen)

In [ ]:
heft = HEFT(G, bw, pp, simple_paths, reciprocals_list, proportions_list, pp_required, data_stream)
start = datetime.datetime.now()
DAGs_orders, DAGs_deploy = heft.get_response_time(sorted_DAG_path=BATCH_TASK_TOPOLOGICAL_ORDER_PATH)
end = datetime.datetime.now()
print('Computer\'s running time:', (end - start).seconds, 'seconds')
print('\nThe finish time of each function on the chosen server for DAG #%d:' % DAG_num_chosen)
pprint.pprint(DAGs_orders[DAG_num_chosen])